In [ ]:
import pandas as pd
import requests
from io import StringIO
from datetime import datetime

In [ ]:
# Get today's date in required format: yyyy_mm_dd
today = datetime.today().strftime('%Y_%m_%d')
print(today)

In [ ]:
# Build filename dynamically
url = f'https://www.kritische-anleger.de/downloads/festgeld_zinsentwicklung_{today}.csv'
print(url)

In [ ]:
# Pretend to be a browser
headers = {
    "User-Agent": "Mozilla/5.0"
}
# Download file
response = requests.get(url, headers=headers)
# Raise error if still blocked
response.raise_for_status()

In [ ]:
# Convert text to DataFrame
df = pd.read_csv(StringIO(response.text), sep=',')

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
# Show all column names
print("Column names:")
print(df.columns.tolist())
column = df.columns.tolist()

# Find columns convertible to date format dd.mm.yyyy
date_columns = []

for col in df.columns:
    try:
        df[col] = pd.to_datetime(df[col], format='%d.%m.%Y', errors='raise')
        date_columns.append(col)
    except:
        pass

print("\nColumns convertible to datetime:")
print(date_columns)

# Require exactly one date column
if len(date_columns) != 1:
    raise ValueError("Expected exactly one date column.")

date_col = date_columns[0]

In [ ]:
# Check all other columns are float-convertible
float_columns = []

for col in df.columns:

    if col == date_col:
        continue

    try:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(['nan', 'NaN', '', 'None', 'null'], pd.NA)
            .str.replace(',', '.', regex=False)   # German decimal comma
        )

        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

        print(f"{col}: converted to float")
        float_columns.append(col)

    except Exception as e:
        raise ValueError(f"Column '{col}' is not float-convertible. {e}")

In [ ]:
# Save to csv termstructure.csv
df.to_csv(
    "termstructure.csv",
    index=False,
    sep=',',
    decimal='.',      # German decimal comma
    # sep=";",          # German CSV separator
    # decimal=",",      # German decimal comma
    # date_format="%d.%m.%Y" # German date format
)